In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
from utils.outlier_detection import detect_outliers_by_series
from utils.formatting_functions import *
from utils.db_functions import *
from utils.excel_functions import *

In [8]:
import os
from pathlib import Path
import pandas as pd

# Read from gdrive or locally
use_gdrive = False

# Data path
if use_gdrive:
    gdrive_path = os.path.join(
        find_shared_folder("ISPAR Team Shared"), "Metadata Automation",
    )
    data_path = Path(gdrive_path) / 'Data'
else:
    data_path = Path('C:/Users/alfon/Desktop/UN/Data')

metadata_output_path = Path(data_path) / 'Metadata Automation Outputs'

SHEET = 'Summary Stats'

# ---------------------------------------------------------------
# 1. Read the 'Summary Stats' sheet of every Excel file
# ---------------------------------------------------------------
tables = {}   # file stem -> DataFrame(Category, Count, Percentage)
errors = {}

for f in sorted(metadata_output_path.glob('*.xlsx')):
    if f.name.startswith('~$'):          # skip Excel lock/temp files
        continue
    try:
        df = pd.read_excel(f, sheet_name=SHEET)
        df.columns = [str(c).strip() for c in df.columns]
        df['Category'] = df['Category'].astype(str).str.strip()
        tables[f.stem] = df[['Category', 'Count', 'Percentage']]
    except Exception as e:                # e.g. sheet missing
        errors[f.stem] = str(e)

if errors:
    print("Files that could not be read:")
    for k, v in errors.items():
        print(f"  {k}: {v}")

# ---------------------------------------------------------------
# 2. Which files share the same categories and which differ
# ---------------------------------------------------------------
cat_sets = {name: tuple(df['Category']) for name, df in tables.items()}

groups = {}
for name, cats in cat_sets.items():
    groups.setdefault(cats, []).append(name)

print("\n=== Category comparison ===")
for i, (cats, files) in enumerate(groups.items(), 1):
    print(f"\nGroup {i} ({len(files)} file(s)): {files}")
    print(f"  Categories: {list(cats)}")

if len(groups) > 1:
    # Show what differs relative to the most common category set
    ref_cats = max(groups, key=lambda k: len(groups[k]))
    print(f"\nDifferences vs. most common set (Group with {len(groups[ref_cats])} files):")
    for cats, files in groups.items():
        if cats == ref_cats:
            continue
        missing = [c for c in ref_cats if c not in cats]
        extra = [c for c in cats if c not in ref_cats]
        print(f"  {files}: missing={missing}, extra={extra}")
else:
    print("\nAll files have identical categories (same order).")

# ---------------------------------------------------------------
# 3. Build combined dataframe
#    rows = file name (no .xlsx), columns = (Category, Count/Percentage)
#    'Total' is the last column pair (it is the last row in each source table)
# ---------------------------------------------------------------
wide = {}
for name, df in tables.items():
    s = df.set_index('Category')
    wide[name] = {}
    for cat in s.index:
        wide[name][(cat, 'Count')] = s.loc[cat, 'Count']
        wide[name][(cat, 'Percentage')] = s.loc[cat, 'Percentage']

result = pd.DataFrame.from_dict(wide, orient='index')
result.columns = pd.MultiIndex.from_tuples(result.columns, names=['Category', 'Metric'])

# Order: categories A..G in order of appearance, Total last
cats_in_order = []
for df in tables.values():
    for c in df['Category']:
        if c != 'Total' and c not in cats_in_order:
            cats_in_order.append(c)
if any('Total' in df['Category'].values for df in tables.values()):
    cats_in_order.append('Total')

# All counts first (A, A1, B, C, ... Total), then all percentages
ordered_cols = [(c, m) for m in ('Count', 'Percentage') for c in cats_in_order
                if (c, m) in result.columns]
result = result[ordered_cols]

# Remove 'Metadata_Automation' from file names (and tidy leftover separators)
result.index = (result.index.str.replace('Metadata_Automation', '', regex=False)
                .str.strip('_- '))
result.index.name = 'File'

pd.set_option('display.width', 250, 'display.max_columns', None)
print("\n=== Combined Summary Stats ===")
print(result)

# Optional: flat version / save
flat = result.copy()
flat.columns = [f"{c}_{m}" for c, m in flat.columns]
# flat.to_csv(metadata_output_path / 'summary_stats_combined.csv')

# ---------------------------------------------------------------
# 4. Combine the 'KPI Data Summary' sheet of every file into one df
#    First column = file name (same cleaned name as above)
# ---------------------------------------------------------------
KPI_SHEET = 'KPI Data Summary'

def clean_name(stem):
    return stem.replace('Metadata_Automation', '').strip('_- ')

kpi_frames = []
kpi_errors = {}

for f in sorted(metadata_output_path.glob('*.xlsx')):
    if f.name.startswith('~$'):
        continue
    try:
        k = pd.read_excel(f, sheet_name=KPI_SHEET)
        k.insert(0, 'File', clean_name(f.stem))
        kpi_frames.append(k)
    except Exception as e:
        kpi_errors[f.stem] = str(e)

if kpi_errors:
    print("\nFiles where 'KPI Data Summary' could not be read:")
    for k_, v in kpi_errors.items():
        print(f"  {k_}: {v}")

kpi_df = pd.concat(kpi_frames, ignore_index=True) if kpi_frames else pd.DataFrame()
print("\n=== Combined KPI Data Summary ===")
print(kpi_df)

# kpi_df.to_csv(metadata_output_path / 'kpi_data_summary_combined.csv', index=False)


=== Category comparison ===

Group 1 (23 file(s)): ['Metadata_Automation_index_10', 'Metadata_Automation_index_1021', 'Metadata_Automation_index_1022', 'Metadata_Automation_index_1024', 'Metadata_Automation_index_1026', 'Metadata_Automation_index_1027', 'Metadata_Automation_index_1030', 'Metadata_Automation_index_1031', 'Metadata_Automation_index_1032', 'Metadata_Automation_index_1034', 'Metadata_Automation_index_1035', 'Metadata_Automation_index_1036', 'Metadata_Automation_index_1037', 'Metadata_Automation_index_1049', 'Metadata_Automation_index_1050', 'Metadata_Automation_index_1051', 'Metadata_Automation_index_1052', 'Metadata_Automation_index_1053', 'Metadata_Automation_index_12', 'Metadata_Automation_index_3', 'Metadata_Automation_index_5', 'Metadata_Automation_index_7', 'Metadata_Automation_index_9']
  Categories: ['A_total', 'A1_cap_adjusted_match', 'B_total', 'C_data_available_no_match', 'D_possible_impute', 'E_no_historical_data', 'F_no_report_data', 'G_no_historical_and_repo

In [9]:
kpi_df

,File,KPI ID,Series Name,n_unique_categorical_simulator,unique_categorical_values_simulator,possible_cap,cap_count,max_kpi_value_hist,possible_floor,floor_count,n_datapoints,A_exact_sum,A_approx_sum,A1_cap_adj_sum,B_exact_sum,B_approx_sum,total_sum,A_sum,B_sum,exact_sum,approx_sum,pct_A,pct_A1_cap_adj,pct_B,pct_exact,pct_approx,pct_matched,year_min,year_max,year_range,year_median,most_common_year,most_common_year_pct,n_distinct_years,year_distribution,year_distribution_formatted
0,index_10,287,CRPD ratification (UN Treaty),NaN,NaN,NaN,NaN,2.50,NaN,NaN,146,136,0,0,0,0,136,136,0,136,0,93.2,0.0,0.0,93.2,0.0,93.2,2020.0,2020.0,0.0,2020.0,2020.0,100.0,1,{2020: 100.0},2020: 100.0%
1,index_10,288,General Law protecting the Rights of Persons w...,2.0,"{0, 5}",NaN,NaN,5.00,NaN,NaN,146,136,0,0,0,0,136,136,0,136,0,93.2,0.0,0.0,93.2,0.0,93.2,2020.0,2020.0,0.0,2020.0,2020.0,100.0,1,{2020: 100.0},2020: 100.0%
2,index_10,289,Definition of ICT accessibility,2.0,"{0, 5}",NaN,NaN,5.00,NaN,NaN,146,136,0,0,0,0,136,136,0,136,0,93.2,0.0,0.0,93.2,0.0,93.2,2020.0,2020.0,0.0,2020.0,2020.0,100.0,1,{2020: 100.0},2020: 100.0%
3,index_10,290,Definition of Reasonable Accommodation,2.0,"{0, 5}",NaN,NaN,5.00,NaN,NaN,146,136,0,0,0,0,136,136,0,136,0,93.2,0.0,0.0,93.2,0.0,93.2,2020.0,2020.0,0.0,2020.0,2020.0,100.0,1,{2020: 100.0},2020: 100.0%
4,index_10,291,Universal Service Obligation includes Persons ...,2.0,"{0, 5}",NaN,NaN,5.00,NaN,NaN,146,136,0,0,0,0,136,136,0,136,0,93.2,0.0,0.0,93.2,0.0,93.2,2020.0,2020.0,0.0,2020.0,2020.0,100.0,1,{2020: 100.0},2020: 100.0%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1168,index_9,274,Active mobile-broadband subscriptions per 100 ...,NaN,NaN,120.0,32.0,246.00,NaN,NaN,193,6,24,20,6,70,126,50,76,32,94,25.9,10.4,39.4,16.6,48.7,65.3,2013.0,2023.0,10.0,2022.0,2022.0,42.1,9,"{2022: 42.1, 2023: 27.8, 2021: 15.1, 2017: 4.8...","2022: 42.1%, 2023: 27.8%, 2021: 15.1%, 2017: 4..."
1169,index_9,279,Adult Literacy (%),NaN,NaN,NaN,NaN,100.00,NaN,NaN,193,13,120,0,0,12,145,133,12,13,132,68.9,0.0,6.2,6.7,68.4,75.1,1980.0,2023.0,43.0,2021.0,2022.0,29.7,21,"{2022: 29.7, 2021: 22.8, 2020: 15.2, 2019: 9.0...","2022: 29.7%, 2021: 22.8%, 2020: 15.2%, 2019: 9..."
1170,index_9,280,Gross Enrollment Ratio,NaN,NaN,NaN,NaN,121.45,NaN,NaN,193,187,0,0,0,2,189,187,2,187,2,96.9,0.0,1.0,96.9,1.0,97.9,2020.0,2022.0,2.0,2022.0,2022.0,98.9,2,"{2022: 98.9, 2020: 1.1}","2022: 98.9%, 2020: 1.1%"
1171,index_9,281,Expected Year of Schooling,NaN,NaN,NaN,NaN,21.08,NaN,NaN,193,180,6,0,0,2,188,186,2,180,8,96.4,0.0,1.0,93.3,4.1,97.4,2012.0,2022.0,10.0,2022.0,2022.0,98.9,3,"{2022: 98.9, 2021: 0.5, 2012: 0.5}","2022: 98.9%, 2021: 0.5%, 2012: 0.5%"
